# 轨迹 vs reachable tube 对比

跑 `compare.py` 的逻辑（几何判断在 `tube_geometry.py`，统计在 `tube_report.py`，画图在 `tube_plot.py`），把 `real_trajectories.npz` / `dwm_trajectories_<variant>.npz` 和 `verify.py` 算出来的 `results/<env>/safety_result.json` 做包含性对比。

**前置条件**：`results/<env>/safety_result.json` 必须已经存在， `verify.py` 要先跑完（本地跑：`mpirun -np <N> python verify.py config/starv_verification/<env>.json`）。

本文件放在 `verifiable_wm/notebooks/` 下，从 `verifiable_wm/` 或 `verifiable_wm/notebooks/` 启动 Jupyter 均可，第一个 cell 会自动定位仓库根目录。

In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "dynamic.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
assert (NOTEBOOK_DIR / "dynamic.py").exists(), (
    f"没找到仓库根目录（dynamic.py），请从 verifiable_wm/ 或 verifiable_wm/notebooks/ 启动 Jupyter"
)
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

In [ ]:
import compare

def run_compare_tube(env_name, variant=None, dataset_name="dataset_v1", **extra_flags):
    """薄封装：把关键字参数拼成 compare.py 的 argv，走跟 CLI 完全一样的代码路径
    （parse_args -> run），不是重新实现一遍逻辑。

    env_name: cartpole / mountain_car / pendulum
    variant: 决定读哪个 dwm_trajectories_<variant>.npz；cartpole 必须显式给
             （intensity / saliency / hybrid），pendulum 默认 'saliency'，
             mountain_car 默认 'old'（见 compare.py 的 DEFAULT_VARIANT）。
    dataset_name: datasets/<env>/data/<dataset_name>/ 下那个版本目录。
    extra_flags: 透传给 compare.py 的其它 CLI flag，用下划线写，比如
                 max_steps=20, only_in_grid=True, check_dims=[0, 1]。
    """
    argv = ["--env", env_name, "--dataset-name", dataset_name]
    if variant is not None:
        argv += ["--variant", variant]

    for key, value in extra_flags.items():
        flag = "--" + key.replace("_", "-")
        if isinstance(value, bool):
            if value:
                argv.append(flag)
        elif isinstance(value, (list, tuple)):
            argv.append(flag)
            argv += [str(v) for v in value]
        else:
            argv += [flag, str(value)]

    args = compare.parse_args(argv)
    compare.run(args)
    return args.outdir

## Case 1: CartPole - saliency（当前主线，alpha=8, lambda_ctrl=0.1）

In [ ]:
outdir = run_compare_tube("cartpole", variant="saliency")
outdir

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(outdir / "real_vs_reachable_tube_examples.png")))
display(Image(filename=str(outdir / "dwm_vs_reachable_tube_examples.png")))

## Case 2: CartPole - intensity（论文 baseline，对照组）

In [ ]:
outdir = run_compare_tube("cartpole", variant="intensity")
outdir

## Case 3: Pendulum - saliency

跑之前确认 `results/pendulum/safety_result.json` 已经由 `verify.py` 生成。

In [ ]:
outdir = run_compare_tube("pendulum", variant="saliency")

from IPython.display import Image, display
display(Image(filename=str(outdir / "real_vs_reachable_tube_examples.png")))
display(Image(filename=str(outdir / "dwm_vs_reachable_tube_examples.png")))

## Case 4: MountainCar（冻结，占位）

MC 的 controller 有问题（occlusion heatmap 关注区域异常），实验内容冻结中，
这里只保留结构占位；decoder 只有一份老权重，variant 默认 `old`。

In [ ]:
# outdir = run_compare_tube("mountain_car")